# Chicago Fed NFCI 历史版本来源预检

仅一次匿名读取官方 ALFRED 下载表单、封存原始 HTML/NAS，核查版本日期范围，不取市场收益、拟合模型或执行账户。

In [1]:
from pathlib import Path
from datetime import datetime,timezone
import hashlib,io,json,os,shlex,subprocess,tarfile,tempfile
import requests
from bs4 import BeautifulSoup
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd();card=root/'research/chicago_nfci_source_preflight_20261023';plan_path=card/'plan.json';plan=json.loads(plan_path.read_text())
assert plan['frozen'] and not (card/'run_state.json').exists()
url=plan['official_source_urls'][0];rawdir=root/'data/snapshots/chicago_nfci_source_preflight_20261023';rawdir.mkdir(parents=True,exist_ok=True)
(card/'run_state.json').write_text(json.dumps({'status':'form_GET_started','plan_sha256':hashlib.sha256(plan_path.read_bytes()).hexdigest(),'GET_attempts':1,'data_ZIP_GETs':0,'model_fits':0,'accounts':0},indent=2)+'\n')
started=datetime.now(timezone.utc).isoformat();response=requests.get(url,timeout=40,headers={'User-Agent':'SOXL-Qlib-source-audit/1.0','Accept':'text/html'});received=datetime.now(timezone.utc).isoformat()
raw=response.content;receipt={'url':url,'requested_at_utc':started,'received_at_utc':received,'http_status':response.status_code,'response_headers':{k:response.headers.get(k) for k in ('Date','Last-Modified','ETag','Content-Type','Content-Length','Cache-Control')},'bytes':len(raw),'sha256':hashlib.sha256(raw).hexdigest()}
(rawdir/'alfred_nfci_form.html').write_bytes(raw);(card/'form_receipt.json').write_text(json.dumps(receipt,indent=2)+'\n')
assert response.status_code==200 and len(raw)<=plan['budget']['max_response_bytes'] and raw.lstrip().lower().startswith(b'<!doctype html')
soup=BeautifulSoup(raw,'html.parser');assert 'Chicago Fed National Financial Conditions Index' in soup.get_text(' ',strip=True)
forms=[f for f in soup.find_all('form') if f.find(attrs={'name':'form[file_type]'}) is not None];assert len(forms)==1
form=forms[0];selector=form.find(attrs={'name':'form[selected_vintage_dates][]'});assert selector
vintages=[o.get('value') for o in selector.find_all('option') if o.get('value')];assert vintages and len(vintages)==len(set(vintages));vintages=sorted(vintages)
start=form.find(attrs={'name':'form[obs_start_date]'}).get('value');end=form.find(attrs={'name':'form[obs_end_date]'}).get('value')
assert vintages[0]<='2011-12-31' and vintages[-1]>='2026-01-01'
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='nfci-form-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:out.add(rawdir/'alfred_nfci_form.html',arcname='alfred_nfci_form.html')
 digest=hashlib.sha256(archive.read_bytes()).hexdigest();size=archive.stat().st_size
 evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 cmd=['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes','-o','ConnectTimeout=5',settings['ssh']['host'],'tar -xOzf '+repr(remote)+' alfred_nfci_form.html']
 restored=subprocess.run(cmd,capture_output=True,timeout=120);assert restored.returncode==0 and hashlib.sha256(restored.stdout).hexdigest()==receipt['sha256'] and len(restored.stdout)==len(raw)
result={'status':'official_ALFRED_NFCI_vintage_form_available_source_only','plan_sha256':hashlib.sha256(plan_path.read_bytes()).hexdigest(),'form_sha256':receipt['sha256'],'observation_start':start,'observation_end':end,'selectable_vintage_dates':len(vintages),'first_vintage_date':vintages[0],'last_vintage_date':vintages[-1],'vintage_covers_training_after_first_date':True,'early_2011_first_public_qualified':False,'nas_archive_sha256':digest,'nas_archive_bytes':size,'nas_artifact':evidence,'nas_remote_hash_verified':True,'scoped_restore_sha256_verified':True,'new_GETs':1,'data_ZIP_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'SOXX_goal_achieved':False}
(card/'source_preflight_result.json').write_text(json.dumps(result,indent=2)+'\n')
(card/'run_state.json').write_text(json.dumps({'status':'completed','plan_sha256':result['plan_sha256'],'GETs':1,'data_ZIP_GETs':0,'model_fits':0,'accounts':0},indent=2)+'\n')
print({k:result[k] for k in ('status','selectable_vintage_dates','first_vintage_date','last_vintage_date','observation_start','observation_end')})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


ReadTimeout: HTTPSConnectionPool(host='alfred.stlouisfed.org', port=443): Read timed out. (read timeout=40)

## 结论边界

版本列表证明可请求性，不证明已拿到每周原始值或交易时点。2011 年早期若无发布版本保持未知；任何 Stage1 信息筛查另行冻结。